In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=2000, n_features=8, n_informative=4, n_redundant=1,
    weights=[0.92, 0.08], flip_y=0.03, class_sep=0.85, random_state=7,
)

columns = ["income", "loan_amount", "age", "months_employed",
           "existing_loans", "credit_score", "debt_to_income", "late_payments_12m"]

df = pd.DataFrame(X, columns=columns)
df["defaulted"] = y

print(df.shape)
print(df["defaulted"].value_counts())
print("default rate:", round(df["defaulted"].mean(), 3))

(2000, 9)
defaulted
0    1813
1     187
Name: count, dtype: int64
default rate: 0.094


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(len(X_train), "training rows |", len(X_test), "test rows")
print("defaulters in test set:", y_test.sum())

1500 training rows | 500 test rows
defaulters in test set: 47


In [4]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print("dummy accuracy:", round(dummy.score(X_test, y_test), 3))

dummy accuracy: 0.906


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

print("model accuracy:", round(model.score(X_test, y_test), 3))

model accuracy: 0.92


In [6]:
from sklearn.metrics import confusion_matrix, classification_report

preds = model.predict(X_test)
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds, target_names=["repaid", "default"], digits=2))

[[451   2]
 [ 38   9]]
              precision    recall  f1-score   support

      repaid       0.92      1.00      0.96       453
     default       0.82      0.19      0.31        47

    accuracy                           0.92       500
   macro avg       0.87      0.59      0.63       500
weighted avg       0.91      0.92      0.90       500



In [7]:
balanced = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)
balanced.fit(X_train, y_train)
preds_b = balanced.predict(X_test)

print("accuracy:", round(balanced.score(X_test, y_test), 3))
print(confusion_matrix(y_test, preds_b))
print(classification_report(y_test, preds_b, target_names=["repaid", "default"], digits=2))

accuracy: 0.776
[[358  95]
 [ 17  30]]
              precision    recall  f1-score   support

      repaid       0.95      0.79      0.86       453
     default       0.24      0.64      0.35        47

    accuracy                           0.78       500
   macro avg       0.60      0.71      0.61       500
weighted avg       0.89      0.78      0.82       500



In [8]:
from sklearn.metrics import precision_score, recall_score

probabilities = model.predict_proba(X_test)[:, 1]

print(f"{'threshold':>10} {'recall':>8} {'precision':>10} {'flagged':>8}")
for t in [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]:
    preds_t = (probabilities >= t).astype(int)
    print(f"{t:>10} {recall_score(y_test, preds_t):>8.2f} "
          f"{precision_score(y_test, preds_t, zero_division=0):>10.2f} {preds_t.sum():>8}")

 threshold   recall  precision  flagged
       0.5     0.19       0.82       11
       0.4     0.26       0.67       18
       0.3     0.32       0.50       30
       0.2     0.45       0.45       47
       0.1     0.57       0.23      118
      0.05     0.70       0.17      195


In [9]:
from sklearn.metrics import confusion_matrix

cost_fn = 100_000   # missing a defaulter
cost_fp = 10_000    # wrongly flagging a good customer

for t in [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]:
    preds_t = (probabilities >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds_t).ravel()
    total_cost = fn * cost_fn + fp * cost_fp
    print(f"threshold {t}: missed={fn}, false alarms={fp}, total cost=₦{total_cost:,}")

threshold 0.5: missed=38, false alarms=2, total cost=₦3,820,000
threshold 0.4: missed=35, false alarms=6, total cost=₦3,560,000
threshold 0.3: missed=32, false alarms=15, total cost=₦3,350,000
threshold 0.2: missed=26, false alarms=26, total cost=₦2,860,000
threshold 0.1: missed=20, false alarms=91, total cost=₦2,910,000
threshold 0.05: missed=14, false alarms=162, total cost=₦3,020,000


In [10]:
from sklearn.tree import DecisionTreeClassifier

deep = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
print("deep tree - train:", round(deep.score(X_train, y_train), 3),
      "| test:", round(deep.score(X_test, y_test), 3))

shallow = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print("depth-3   - train:", round(shallow.score(X_train, y_train), 3),
      "| test:", round(shallow.score(X_test, y_test), 3))

deep tree - train: 1.0 | test: 0.902
depth-3   - train: 0.943 | test: 0.922


Why does limiting depth help on new data? Because it stops the tree from memorising noise and forces it to learn only the patterns that hold across many loans, and those are the ones that appear in new data too.
If you only reported training accuracy, which model would you have shipped? The deep tree (100% beats 94.3%). That would have been the worse model on real data. This is why you must always evaluate on a held-out test set.



In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

threshold = 0.2
results = {
    "Dummy": dummy.predict(X_test),
    "Logistic regression": model.predict(X_test),
    "Logistic, class-weighted": balanced.predict(X_test),
    f"Logistic, threshold {threshold}": (model.predict_proba(X_test)[:, 1] >= threshold).astype(int),
    "Decision tree, depth 3": shallow.predict(X_test),
}

for name, p in results.items():
    print(f"\n{name}")
    print(confusion_matrix(y_test, p))
    print(f"accuracy={accuracy_score(y_test, p):.3f}  "
          f"recall={recall_score(y_test, p):.2f}  "
          f"precision={precision_score(y_test, p, zero_division=0):.2f}  "
          f"caught={int(((p == 1) & (y_test == 1)).sum())} of {int(y_test.sum())}")


Dummy
[[453   0]
 [ 47   0]]
accuracy=0.906  recall=0.00  precision=0.00  caught=0 of 47

Logistic regression
[[451   2]
 [ 38   9]]
accuracy=0.920  recall=0.19  precision=0.82  caught=9 of 47

Logistic, class-weighted
[[358  95]
 [ 17  30]]
accuracy=0.776  recall=0.64  precision=0.24  caught=30 of 47

Logistic, threshold 0.2
[[427  26]
 [ 26  21]]
accuracy=0.896  recall=0.45  precision=0.45  caught=21 of 47

Decision tree, depth 3
[[449   4]
 [ 35  12]]
accuracy=0.922  recall=0.26  precision=0.75  caught=12 of 47


In [12]:
test_df = pd.DataFrame(X_test, columns=columns)
test_df["actual"] = y_test
test_df["predicted"] = balanced.predict(X_test)
test_df["probability"] = balanced.predict_proba(X_test)[:, 1]

missed = test_df[(test_df["actual"] == 1) & (test_df["predicted"] == 0)]
print(len(missed), "defaulters missed")
print(missed.sort_values("probability", ascending=False).head(10))

caught = test_df[(test_df["actual"] == 1) & (test_df["predicted"] == 1)]
comparison = pd.DataFrame({"missed": missed[columns].mean(),
                           "caught": caught[columns].mean()})
comparison["difference"] = comparison["missed"] - comparison["caught"]
print(comparison.round(2))

17 defaulters missed
       income  loan_amount       age  months_employed  existing_loans  \
335  2.044454     0.209550  0.069818        -0.326909        0.350912   
116 -0.269323     1.299351 -0.288072        -3.399208        2.028629   
433  0.068840     1.376970 -0.016365        -3.587908       -1.686218   
486 -0.578013     0.097075  0.598651        -0.214851       -1.057215   
58   1.764601     0.477137  0.815408        -1.811411       -1.101770   
248 -0.410474     1.099338  1.552991         0.810463       -1.823105   
179 -0.346579    -1.508905  0.417276         0.028676        1.262860   
167 -1.874132    -1.277142  0.814503        -0.326278       -0.533810   
497 -0.735079    -0.551122  1.498643         1.721922        1.649005   
323 -0.364610    -0.370659  0.885062        -1.269634        0.562126   

     credit_score  debt_to_income  late_payments_12m  actual  predicted  \
335      2.241093       -0.672653           1.818542       1          0   
116      0.829835       -

In [13]:
print(missed["probability"].describe().round(2))
print((missed["probability"] >= 0.3).sum(), "of 17 missed have probability >= 0.3")

count    17.00
mean      0.23
std       0.12
min       0.07
25%       0.13
50%       0.20
75%       0.30
max       0.47
Name: probability, dtype: float64
4 of 17 missed have probability >= 0.3


Most of the 17 missed defaulters were not near misses. Only 4 had probabilities between 0.38 and 0.47, close to the 0.5 cutoff, so a lower threshold could catch them. The remaining 13 scored 0.30 or below, and the model found them largely unremarkable. Lowering the threshold would not fix these without flagging many more good customers, and 95 are already wrongly flagged. These misses need a different fix: better features, more data, or a model that can learn more than one pattern.

The model is systematically blind to older defaulters with more late payments. Its caught defaulters average age -1.08 while the missed ones average +1.04, a gap of 2.1 standard deviations, the largest of any feature. The model has learned that risk goes with being young, so defaulters who don’t fit that profile slip through. This suggests the defaulters are not one uniform group. There may be at least two different kinds, and a single linear model can only learn one of them well.

## Results Table

**Test set:** 500 loans, 47 of which are real defaulters. All models are evaluated on the same test set.

| Model | Accuracy | Default recall | Default precision | Defaulters caught | Missed | False alarms | Total cost (₦) |
|---|---|---|---|---|---|---|---|
| Dummy (predicts repaid) | 0.906 | 0.00 | none (never flags) | 0 of 47 | 47 | 0 | 4.70M |
| Logistic regression | 0.920 | 0.19 | 0.82 | 9 of 47 | 38 | 2 | 3.82M |
| Logistic, class-weighted | 0.776 | 0.64 | 0.24 | 30 of 47 | 17 | 95 | 2.65M |
| Logistic, threshold 0.2 | 0.896 | 0.45 | 0.45 | 21 of 47 | 26 | 26 | 2.86M |
| Decision tree, depth 3 | 0.922 | 0.26 | 0.75 | 12 of 47 | 35 | 4 | 3.54M |

**Cost assumption:** a missed defaulter loses ₦100,000 of principal and a false alarm (a good customer wrongly flagged) loses about ₦10,000 in interest and future business. Total cost = (missed × ₦100,000) + (false alarms × ₦10,000).

**Key takeaways:**
- The two highest-accuracy models (logistic regression 0.920, depth-3 tree 0.922) catch the fewest defaulters of the real models, so they are the least useful.
- The class-weighted model has the lowest accuracy but the lowest total cost, because it catches 30 of 47 defaulters.
- The threshold-0.2 model is a close second and is more balanced (45% precision, only 26 false alarms).

**Recommendation:** deploy the class-weighted logistic regression under this cost assumption. If the lender can only review about 50 flagged applications, use the threshold-0.2 model instead.